# Layer Inspection and Visualization

This notebook demonstrates how to inspect and visualize S-57 ENC layers using the maritime toolkit for diagnostic and exploratory analysis.

#### Purpose

- **Verify Data**: Check what layers exist in your converted ENC data
- **Validate Coverage**: Ensure expected features are present before graph construction
- **Explore Features**: Understand S-57 object types (buoys, lights, wrecks, etc.)
- **Geographic Filtering**: Query ENCs by area of interest and usage band

#### Workflow Overview

1. **Configuration** - Select backend (PostGIS/GeoPackage) and visualization parameters
2. **Database Connection** - Connect to ENC data source and load base route
3. **Buffer Creation** - Create geographic zone around route for filtering
4. **Usage Band Query** - List available ENCs by scale (overview to berthing)
5. **Geographic Filtering** - Find ENCs intersecting the buffer boundary
6. **Layer Visualization** - Display specific layers on interactive map

#### Data Flow

```
Backend Connection → Route Load → Buffer → ENC Filtering → Layer Query → Interactive Map
```

#### Expected Outputs

- **Interactive Maps**: Plotly visualizations with multiple S-57 layers
- **ENC Inventory**: Charts grouped by usage band and geographic area
- **Layer Coverage**: Visual validation of data quality and feature availability

#### Required Data

This notebook requires:
1. **ENC Data**: S-57 charts converted to PostGIS or GeoPackage format
2. **Base Route**: Pre-computed route for defining area of interest (optional)
3. **PostgreSQL** (if using PostGIS): Database credentials in `.env` file
4. **Mapbox Token** (optional): For custom map tiles in visualizations

**Setup Instructions:** See `docs/SETUP.md`
**Troubleshooting:** See `docs/TROUBLESHOOTING.md`

## 1. Configuration

Select your data backend (PostGIS or GeoPackage), choose the visualization parameters (buffer distance around your route), and specify which S-57 layers to inspect and display. These settings control which data source is queried, how the geographic area of interest is defined, and what features will be visualized in the interactive maps below.

In [ ]:
# =============================================================================
# CONFIGURATION - Adjust these settings before running
# =============================================================================

# --- Backend Selection ---
# Options: "postgis" or "geopackage"
# - "postgis": Best for large datasets, complex spatial queries, and production.
#              Requires a running PostgreSQL database with PostGIS extension.
# - "geopackage": Best for portability, local development, and smaller datasets.
#                 Self-contained in a single .gpkg file (SQLite-based).
BACKEND: str = "geopackage"

# --- PostGIS Configuration (used if BACKEND="postgis") ---
pg_schema: str = "enc_west"

# --- GeoPackage Configuration (used if BACKEND="geopackage") ---
gpkg_file: str = "enc_west.gpkg"  # Path relative to notebook directory

# --- Route Configuration ---
route_schema: str = "routes"  # Only used for PostGIS
route_table: str = "base_routes"
route_name: str = "base_route_gpkg"

# --- Visualization ---
# Buffer distance around route in nautical miles
route_buffer_nm: float = 24.0

# --- Layer Selection ---
# Example layers to visualize (try: boyspp, boylat, bcnlat, lndare, seaare)
example_layers: list[str] = ["seaare", "lndare", "boylat"]

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================

print("=" * 70)
print("✓ Configuration Loaded")
print("=" * 70)
print(f"Backend: {BACKEND.upper()}")
if BACKEND == "postgis":
    print(f"Database schema: {pg_schema}")
elif BACKEND == "geopackage":
    print(f"GeoPackage file: {gpkg_file}")
print(f"Route: {route_table}.{route_name}")
print(f"Buffer distance: {route_buffer_nm} nautical miles")
print(f"Example layers: {', '.join(example_layers)}")
print("=" * 70)

### 1.1 Imports & Environment Setup

In [ ]:
# =============================================================================
# SETUP AND IMPORTS
# =============================================================================

# --- Standard Library ---
import sys
import os
from pathlib import Path

# --- Environment Setup ---
from dotenv import load_dotenv
import plotly.io as pio
import plotly.graph_objects as go

# --- Fix PROJ_LIB Path (Common Conda/Jupyter Issue) ---
# Ensure GDAL/PROJ can find the coordinate database
conda_prefix = sys.prefix
possible_proj_lib = os.path.join(conda_prefix, 'share', 'proj')
if os.path.exists(possible_proj_lib):
    os.environ['PROJ_LIB'] = possible_proj_lib

# --- Project Setup ---
# Define the project root as two directories up from the current notebook.
# This is a robust way to set the path in a known project structure.
notebook_dir = Path.cwd()
project_root = notebook_dir.parent.parent # Assumes notebook is in /docs/notebooks
load_dotenv(project_root / ".env")
pio.renderers.default = "notebook_connected"

# --- Project Imports ---
from nautical_graph_toolkit.core.s57_data import ENCDataFactory
from nautical_graph_toolkit.utils.plot_utils import PlotlyChart
from nautical_graph_toolkit.utils.s57_utils import S57Utils
from nautical_graph_toolkit.utils.geometry_utils import Buffer

print("=" * 70)
print("✓ All imports loaded successfully")
print("=" * 70)

### 1.2 Environment Validation

In [ ]:
if BACKEND == "postgis":
    # PostGIS backend
    db_params = {
        'dbname': os.getenv('DB_NAME'),
        'user': os.getenv('DB_USER'),
        'password': os.getenv('DB_PASSWORD'),
        'host': os.getenv('DB_HOST'),
        'port': os.getenv('DB_PORT')
    }

    # Validate environment
    required_env_vars = ['DB_NAME', 'DB_USER', 'DB_PASSWORD', 'DB_HOST', 'DB_PORT']
    missing_vars = [v for v in required_env_vars if not os.getenv(v)]

    if missing_vars:
        print(f"❌ Missing environment variables: {', '.join(missing_vars)}")
        raise ValueError("Cannot proceed without database configuration")

    print(f"Backend: PostGIS")
    print(f"Database: {db_params['dbname']}@{db_params['host']}")

elif BACKEND == "geopackage":
    # GeoPackage backend
    # Construct path relative to the project root for robustness
    data_file = project_root / "data" / gpkg_file

    # Validate file exists
    if not data_file.exists():
        print(f"❌ GeoPackage file not found: {data_file}")
        print(f"   Please ensure the file exists or update gpkg_file configuration")
        raise FileNotFoundError(f"Cannot proceed without GeoPackage file")

    print(f"Backend: GeoPackage")
    print(f"File: {data_file.name}")
    print(f"Size: {data_file.stat().st_size / (1024**2):.2f} MB")

else:
    raise ValueError(f"Invalid BACKEND: {BACKEND}. Must be 'postgis' or 'geopackage'")

# --- Output Directory ---
output_dir = notebook_dir / 'output'
output_dir.mkdir(exist_ok=True)

# Initialize utilities
s57_utils = S57Utils()
ply = PlotlyChart()

mapbox_token = os.getenv('MAPBOX_TOKEN')
if not mapbox_token:
    print("⚠️  MAPBOX_TOKEN not configured - using default basemap")
else:
    print("✓ Mapbox token loaded")

print("=" * 70)

### 1.3 Workflow Context

**Purpose**: Diagnostic and exploratory tool for S-57 ENC layer inspection and visualization

**When to Use This Notebook**:
- **Before graph creation**: Verify what ENC data is available in your area of interest
- **Debugging data issues**: Check if expected layers/features exist in your ENC dataset
- **Understanding layer coverage**: Visualize which layers are present at different usage bands (scales)
- **Area validation**: Create buffer zones and explore what ENCs intersect your route
- **Feature exploration**: Understand S-57 object types (buoys, wrecks, lights, etc.) before routing

**Integration with Main Pipeline**:
This notebook is NOT part of the main routing pipeline but supports all three main steps:
- **Step 1** (Import): Verify converted ENC data is accessible and properly formatted
- **Step 2** (Graph Construction): Explore which layers should be used for navigable areas
- **Step 3** (Weighting): Understand which feature layers affect routing weights

**Key Features**:
- Works with **PostGIS** or **GeoPackage** backends (configure `BACKEND` setting)
- Queries by geographic boundary (route buffer)
- Visualizes ENCs by usage band (scale levels 1-6)
- Displays arbitrary S-57 layers on interactive map
- Supports both line/point features (buoys, wrecks, lights) and polygon features (sea areas, land)

**Output**:
- Interactive Plotly maps with multiple layers
- ENC inventory by usage band and geographic area
- Visual validation of data quality and coverage

**Next Steps**:
- Use findings to inform layer selection in `graph_config.yml`
- Proceed to main pipeline (`import_s57.ipynb` → `graph_*_v2.ipynb` → `graph_weighted_directed_*_v2.ipynb`)

## 2. Initialize Database Connection and Load Route

First, connect to the ENC database and load a pre-defined route.
This route will be used to create a geographic buffer for filtering ENCs.

In [ ]:
try:
    # Initialize factory based on backend
    if BACKEND == "postgis":
        data_factory = ENCDataFactory(source=db_params, schema=pg_schema)
        print(f"✓ Connected to PostGIS schema: {pg_schema}")
    elif BACKEND == "geopackage":
        data_factory = ENCDataFactory(source=data_file)
        print(f"✓ Connected to GeoPackage: {data_file.name}")

except Exception as e:
    print(f"❌ Failed to initialize data factory: {e}")
    raise

try:
    # Load route (backend-aware)
    if BACKEND == "postgis":
        base_route = data_factory.load_route(
            route_name=route_name,
            schema_name=route_schema,
            table_name=route_table
        )
    elif BACKEND == "geopackage":
        # GeoPackage doesn't use schema_name
        base_route = data_factory.load_route(
            route_name=route_name,
            table_name=route_table
        )

    # base_route is a Shapely LineString, not a GeoDataFrame
    if base_route is None or base_route.is_empty:
        raise ValueError(f"Route '{route_name}' not found or is empty")

    # Convert route length from degrees to nautical miles
    length_nm = Buffer._degrees_to_nm(base_route.length)

    print(f"✓ Loaded route '{route_name}' successfully")
    print(f"  - Coordinates: {len(base_route.coords)} points")
    print(f"  - Length: {length_nm:.2f} nautical miles")
    print(f"  - Bounds: {base_route.bounds}")

except Exception as e:
    print(f"❌ Failed to load route: {e}")
    print(f"   Please verify:")
    print(f"   - Route name: {route_name}")
    if BACKEND == "postgis":
        print(f"   - Route schema: {route_schema}")
    print(f"   - Route table: {route_table}")
    raise

### 2.1 Create Buffer Zone and Visualize Route

Create a geographic buffer around the route and display it on an interactive map.
This buffer defines the area of interest for layer filtering.

In [ ]:
try:
    route_buffer = Buffer.create_buffer(base_route, route_buffer_nm)
    print(f"✓ Created buffer around route: {route_buffer_nm} nautical miles")
except Exception as e:
    print(f"❌ Failed to create buffer: {e}")
    raise

try:
    # Create map
    ply_fig = ply.create_base_map(mapbox_token=mapbox_token)
    ply.plotly_base_config(ply_fig)

    # Add route buffer to map
    ply.add_polygon_trace(
        fig=ply_fig,
        polygon=route_buffer,
        name="Route Buffer",
        color='blue',
    )

    print("✓ Visualization created successfully")
    ply_fig.show()

except Exception as e:
    print(f"❌ Failed to create visualization: {e}")
    raise

## 3. Query ENCs by Usage Band

S-57 charts are classified by usage band from overview (band 1) to berthing (band 6).
These bands represent different scales and levels of detail. For the area of interest
(defined by your route buffer), query which ENCs are available at each scale.

In [ ]:
try:
    # Query usage bands
    usage_band_names = {
        1: "Overview (1:2,5M+)",
        2: "General (1:640K-1,280M)",
        3: "Coastal (1:160K-320K)",
        4: "Approach (1:40K-80K)",
        5: "Harbour (1:10K-20K)",
        6: "Berthing (<1:5K)"
    }

    enc_by_band = {}
    print("=" * 70)
    print("Available ENCs by Usage Band:")
    print("=" * 70)

    for band in range(1, 7):
        enc_by_band[band] = data_factory.get_encs_by_usage_band(usage_band=band)
        count = len(enc_by_band[band])
        print(f"Band {band}: {usage_band_names[band]:<25} {count:>6} ENCs")

    print("=" * 70)

except Exception as e:
    print(f"❌ Failed to query usage bands: {e}")
    raise

## 4. Filter ENCs by Geographic Boundary

Using the route buffer created in Step 2, filter the complete ENC dataset to find
which charts intersect with your area of interest. This focuses subsequent layer
queries to the relevant charts only.

In [ ]:
try:
    encs_in_buffer = data_factory.get_encs_by_boundary(route_buffer)

    if not encs_in_buffer:
        print("⚠️  No ENCs found within the buffer boundary")
        print("   Consider increasing route_buffer_nm value")
    else:
        print(f"✓ Found {len(encs_in_buffer)} ENCs intersecting the buffer boundary")
        print(f"  Example ENCs: {', '.join(encs_in_buffer[:3])}")

except Exception as e:
    print(f"❌ Failed to filter ENCs by boundary: {e}")
    raise

## 5. Retrieve and Visualize Layers

Select specific S-57 layers (object classes) and display them on the map.
Each layer represents a feature type (sea areas, buoys, lights, dangers, etc.)
from the filtered ENC dataset.

**Common Layers**:
- **seaare**: Sea areas (open water regions)
- **lndare**: Land areas (coastline and islands)
- **boyspp**: Buoys in special position
- **bcnlat**: Lateral beacons (channel markers)
- **wreck**: Wreck positions
- **soundg**: Sounding points (depth measurements)

In [ ]:
try:
    # Create fresh figure for layer visualization
    ply_fig_layers = ply.create_base_map(mapbox_token=mapbox_token)
    ply.plotly_base_config(ply_fig_layers)

    # Add route buffer as reference
    ply.add_polygon_trace(
        fig=ply_fig_layers,
        polygon=route_buffer,
        name="Route Buffer",
        color='lightblue',

    )

    # Visualize each configured layer
    layer_colors = {
        "seaare": "lightcyan",
        "lndare": "tan",
        "boyspp": "red",
        "bcnlat": "orange",
        "wreck": "purple",
        "soundg": "green"
    }

    for layer_name in example_layers:
        if not encs_in_buffer:
            print(f"⚠️  Skipping {layer_name}: no ENCs in buffer")
            continue

        try:
            layer_data = data_factory.get_layer(
                layer_name=layer_name,
                filter_by_enc=encs_in_buffer
            )

            if layer_data is None or layer_data.empty:
                print(f"⚠️  Layer '{layer_name}': no features found")
                continue

            # Get layer display name
            layer_display_name = s57_utils.get_object_class_name(layer_name)
            print(f"✓ Layer '{layer_name}': {len(layer_data)} features")

            # Add layer to map
            color = layer_colors.get(layer_name, "blue")
            ply.add_layer_trace(
                figure=ply_fig_layers,
                layer_df=layer_data,
                name=layer_display_name,
                color=color
            )

        except Exception as e:
            print(f"⚠️  Failed to load layer '{layer_name}': {e}")

    print("\n✓ Layer visualization complete")
    ply_fig_layers.show()

except Exception as e:
    print(f"❌ Failed to create layer visualization: {e}")
    raise